In [ ]:
%pip install scikit-posthocs seaborn openpyxl --q

# Avaliação de Equivalência dos Rankings — por classe

Lê todos os `Ranking_*.xlsx` (formato padrão: `Feature`, `Score`, `Ranking`) e faz três
coisas: Friedman + Nemenyi (quais rankings são estatisticamente equivalentes), ranking
médio das variáveis, e mapa de calor de equivalência.

**Organização por classe**: o notebook 2 gera os rankings de cada classe numa subpasta própria (`Classe_<rótulo>/`). Este notebook detecta essas subpastas e repete a mesma análise de equivalência uma vez para cada classe, salvando os resultados dentro da própria subpasta (`Classe_<rótulo>/Resultados_Completos_Rankings/`). Se não encontrar nenhuma subpasta `Classe_*/`, analisa a pasta atual como um único cenário.

Se não encontrar nenhuma subpasta `Classe_*/` (por exemplo, no fluxo binário antigo,
com um único cenário numa pasta só), o notebook cai de volta no comportamento de
sempre: analisa a pasta atual.

*Leitura correta do resultado: p > 0,05 significa que o teste **não encontrou
diferença** entre os dois rankings; não prova que eles sejam idênticos.*

### 1. Configurações e descoberta das subpastas de classe

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scikit_posthocs as sp
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy.stats import friedmanchisquare
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

PASTA_RAIZ = os.getcwd()
ALPHA = 0.05  # nível de significância usado em todo o notebook

# As versões "_BaseCompleta" das descritivas usam a base inteira (todas as classes),
# então NÃO são pareadas com o CEQ. Por padrão ficam de fora de tudo. Mude para True
# para incluí-las.
INCLUIR_BASE_COMPLETA = False

FAIXAS = [0.0, 0.05, 0.25, 0.50, 1.0001]
ROTULOS_FAIXAS = [
    "Diferentes (p ≤ 0,05)",
    "Pouco equivalentes (0,05 < p ≤ 0,25)",
    "Moderadamente equivalentes (0,25 < p ≤ 0,50)",
    "Muito equivalentes (p > 0,50)",
]
CORES_FAIXAS = ["#d73027", "#fdae61", "#a6d96a", "#1a9850"]

REGEX_CEQ = re.compile(r"^(Cp|Cpk|Pp|Ppk|p|np|c|u)_(Original|Normalizado)$")
PREFIXOS_DESCRITIVAS = ("AmplitudeSigma", "IQRSigma", "Assimetria", "Curtose")


def grupo_metodo(nome):
    """Classifica o método: 'CEQ', 'Descritiva' ou 'Literatura'."""
    if REGEX_CEQ.match(nome):
        return "CEQ"
    if nome.startswith(PREFIXOS_DESCRITIVAS):
        return "Descritiva"
    return "Literatura"


def is_alvo(nome):
    return grupo_metodo(nome) == "CEQ"


COR_GRUPO = {"CEQ": "#1f77b4", "Descritiva": "#2ca02c", "Literatura": "#333333"}


def descobrir_escopos(pasta_raiz):
    """Descobre as subpastas Classe_<rótulo>/ (saída do notebook 2) dentro de
    `pasta_raiz`. Se nenhuma for encontrada, devolve [(pasta_raiz, None)] -- um único
    escopo, a própria pasta atual (compatibilidade com o fluxo de uma única
    base/classe, como no uso binário original)."""
    subpastas = sorted(
        d for d in os.listdir(pasta_raiz)
        if d.startswith("Classe_") and os.path.isdir(os.path.join(pasta_raiz, d))
    )
    if not subpastas:
        return [(pasta_raiz, None)]
    return [(os.path.join(pasta_raiz, d), d[len("Classe_"):]) for d in subpastas]


ESCOPOS = descobrir_escopos(PASTA_RAIZ)
if ESCOPOS[0][1] is None:
    print(f"Nenhuma subpasta 'Classe_*' encontrada -- analisando a pasta atual como um único cenário: {PASTA_RAIZ}")
else:
    print(f"{len(ESCOPOS)} subpasta(s) de classe encontrada(s): {[c for _, c in ESCOPOS]}")


### 2. Análise completa de um escopo

A função `processar_escopo` reúne a leitura dos rankings, a exportação das tabelas de apêndice, o teste de Friedman, o pós-teste de Nemenyi, o ranking médio das variáveis e o mapa de calor. `FOLDER` e `OUT_DIR` apontam para a subpasta da classe em processamento.

In [ ]:
def processar_escopo(FOLDER, classe_rotulo):
    rotulo_exibicao = f"Classe {classe_rotulo}" if classe_rotulo else "(pasta única)"
    OUT_DIR = os.path.join(FOLDER, "Resultados_Completos_Rankings")
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"\n{'=' * 70}\n {rotulo_exibicao}  ->  {FOLDER}\n{'=' * 70}")
    print(f"Salvando resultados em: {OUT_DIR}")

    # --- 2.1 Leitura dos rankings e construção das tabelas ---
    arquivos_encontrados = sorted(
        f for f in os.listdir(FOLDER) if f.startswith("Ranking_") and f.endswith(".xlsx")
    )
    if not INCLUIR_BASE_COMPLETA:
        ignorados = [f for f in arquivos_encontrados if "_BaseCompleta" in f]
        arquivos_encontrados = [f for f in arquivos_encontrados if "_BaseCompleta" not in f]
        if ignorados:
            print(f"{len(ignorados)} arquivo(s) \'_BaseCompleta\' fora da análise (INCLUIR_BASE_COMPLETA=False).")

    if not arquivos_encontrados:
        print(f"Nenhum arquivo \'Ranking_*.xlsx\' encontrado em {FOLDER} -- pulando este escopo.")
        return

    print(f"Lendo {len(arquivos_encontrados)} arquivo(s) para gerar tabelas e estatísticas...")

    dict_para_estatistica = {}
    dict_tabela_rankings = {}
    dict_tabela_metricas = {}

    for arquivo in arquivos_encontrados:
        path_arq = os.path.join(FOLDER, arquivo)
        try:
            raw_df = pd.read_excel(path_arq)
            raw_df.columns = raw_df.columns.astype(str).str.strip()

            if {"Feature", "Score", "Ranking"}.issubset(raw_df.columns):
                df_temp = raw_df[["Feature", "Score", "Ranking"]].copy()
            elif raw_df.shape[1] >= 3:
                df_temp = raw_df.iloc[:, [0, 1, 2]].copy()
                df_temp.columns = ["Feature", "Score", "Ranking"]
            else:
                print(f"\'{arquivo}\' não tem colunas suficientes. Ignorado.")
                continue

            df_temp["Feature"] = df_temp["Feature"].astype(str).str.strip()
            df_temp["Score"] = pd.to_numeric(df_temp["Score"], errors="coerce")
            df_temp["Ranking"] = pd.to_numeric(df_temp["Ranking"], errors="coerce")
            df_temp = df_temp.dropna(subset=["Feature", "Ranking"])
            df_temp = df_temp.sort_values(by="Ranking")

            nome_limpo = arquivo.replace(".xlsx", "").replace("Ranking_", "")

            dict_tabela_rankings[nome_limpo] = pd.Series(df_temp["Feature"].values)

            df_temp = df_temp.drop_duplicates(subset=["Feature"], keep="first")
            dict_tabela_metricas[nome_limpo] = pd.Series(df_temp["Score"].values, index=df_temp["Feature"].values)
            dict_para_estatistica[nome_limpo] = pd.Series(df_temp["Ranking"].values, index=df_temp["Feature"].values)

        except Exception as e:
            print(f"[ERRO] Falha ao ler {arquivo}: {e}")

    print(f"{len(dict_para_estatistica)} ranking(s) processado(s) com sucesso.")
    contagem = pd.Series({m: grupo_metodo(m) for m in dict_para_estatistica}).value_counts()
    print("Rankings por grupo:", contagem.to_dict())

    # --- 2.2 Exportar tabelas para Excel (apêndice) ---
    df_tabela_rankings = pd.DataFrame(dict_tabela_rankings)
    df_tabela_rankings.index = range(1, len(df_tabela_rankings) + 1)
    df_tabela_rankings.index.name = "Posição (Ranking)"
    df_tabela_rankings.to_excel(os.path.join(OUT_DIR, "1_Tabela_Apendice_Rankings_Ordenados.xlsx"))

    df_tabela_metricas = pd.DataFrame(dict_tabela_metricas)
    df_tabela_metricas.index.name = "Atributo (Feature)"
    df_tabela_metricas.to_excel(os.path.join(OUT_DIR, "2_Tabela_Apendice_Metricas_Scores.xlsx"))
    print("Tabelas Excel de apêndice geradas.")

    # --- 2.3 Teste de Friedman ---
    df_matrix_stat = pd.DataFrame(dict_para_estatistica).dropna()
    if df_matrix_stat.shape[0] < 3 or df_matrix_stat.shape[1] < 3:
        print(f"Dados insuficientes para estatística em {rotulo_exibicao} (menos de 3 atributos em "
              f"comum ou menos de 3 métodos). Pulando o restante da análise deste escopo.")
        return

    print(f"Atributos em comum a todos os rankings: {df_matrix_stat.shape[0]}")
    print(f"Rankings comparados: {df_matrix_stat.shape[1]}")

    stat, p_friedman = friedmanchisquare(*[df_matrix_stat[col] for col in df_matrix_stat.columns])
    print(f"\nEstatística de Friedman: {stat:.4f} | P-valor: {p_friedman:.4e}")
    if p_friedman <= ALPHA:
        print(f"CONCLUSÃO: p <= {ALPHA}. Há diferença estatística entre pelo menos um par de rankings.")
    else:
        print(f"CONCLUSÃO: p > {ALPHA}. O Friedman não encontrou diferença entre os rankings.")

    # --- 2.4 Post-hoc de Nemenyi ---
    p_valores_nemenyi = sp.posthoc_nemenyi_friedman(df_matrix_stat)

    ordem_grupos = {"CEQ": 0, "Descritiva": 1, "Literatura": 2}
    modelos_ordenados = sorted(df_matrix_stat.columns, key=lambda m: (ordem_grupos[grupo_metodo(m)], m))
    p_valores_nemenyi = p_valores_nemenyi.loc[modelos_ordenados, modelos_ordenados]

    alvos = [m for m in modelos_ordenados if is_alvo(m)]
    rho_spearman = df_matrix_stat.corr(method="spearman").loc[modelos_ordenados, modelos_ordenados]
    print("Matriz de p-valores do Nemenyi calculada.")

    linhas = []
    cols = p_valores_nemenyi.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            m1, m2 = cols[i], cols[j]
            p_valor = float(p_valores_nemenyi.loc[m1, m2])
            faixa = ROTULOS_FAIXAS[np.digitize(p_valor, FAIXAS[1:-1], right=True)]
            linhas.append({
                "Método A": m1, "Grupo A": grupo_metodo(m1),
                "Método B": m2, "Grupo B": grupo_metodo(m2),
                "p-valor": round(p_valor, 4),
                "Spearman ρ": round(float(rho_spearman.loc[m1, m2]), 3),
                "Equivalentes (p > 0,05)": p_valor > ALPHA,
                "Faixa": faixa,
            })

    df_pares = pd.DataFrame(linhas).sort_values("p-valor", ascending=False).reset_index(drop=True)
    df_equivalentes = df_pares[df_pares["Equivalentes (p > 0,05)"]].reset_index(drop=True)

    df_pares.to_excel(os.path.join(OUT_DIR, "4_Todos_os_Pares_Nemenyi.xlsx"), index=False)
    df_equivalentes.to_excel(os.path.join(OUT_DIR, "4_Pares_Estatisticamente_Equivalentes.xlsx"), index=False)

    print(f"\n{len(df_equivalentes)} de {len(df_pares)} pares são estatisticamente equivalentes (p > {ALPHA}).")
    print(df_pares["Faixa"].value_counts().reindex(ROTULOS_FAIXAS).fillna(0).astype(int).to_string())

    RHO_MIN = 0.5
    suspeitos = df_equivalentes[df_equivalentes["Spearman ρ"] < RHO_MIN]
    print(f"\n{len(suspeitos)} de {len(df_equivalentes)} pares \"equivalentes\" têm ρ de Spearman < {RHO_MIN}.")

    # --- 2.5 Resumo: com quem cada métrica do CEQ é equivalente ---
    print(f"\nRankings equivalentes a cada métrica do CEQ (p > {ALPHA}):\n")
    for alvo in alvos:
        desc, lit, ceq = [], [], []
        for outro in modelos_ordenados:
            if outro == alvo:
                continue
            p_valor = float(p_valores_nemenyi.loc[alvo, outro])
            if p_valor > ALPHA:
                item = f"{outro} (p={p_valor:.2f})"
                {"Descritiva": desc, "Literatura": lit, "CEQ": ceq}[grupo_metodo(outro)].append(item)
        print(f"• {alvo}")
        print(f"    CEQ:        {', '.join(ceq) if ceq else '—'}")
        print(f"    Descritiva: {', '.join(desc) if desc else '—'}")
        print(f"    Literatura: {', '.join(lit) if lit else '—'}")

    # --- 2.6 Ranking médio das variáveis ---
    media_posicoes = df_matrix_stat.mean(axis=1)
    df_ranking_medio = (
        pd.DataFrame({"Variável": media_posicoes.index, "Média das posições": media_posicoes.values})
        .sort_values(["Média das posições", "Variável"], ascending=[True, True])
        .reset_index(drop=True)
    )
    df_ranking_medio.insert(0, "Colocação", range(1, len(df_ranking_medio) + 1))
    df_ranking_medio["Média das posições"] = df_ranking_medio["Média das posições"].round(4)

    path_medio = os.path.join(OUT_DIR, "5_Ranking_Medio_Variaveis.xlsx")
    df_ranking_medio.to_excel(path_medio, index=False)
    print(f"\nRanking médio salvo em: {path_medio}")
    print(df_ranking_medio.head(10).to_string(index=False))

    # --- 2.7 Mapa de calor de equivalência (Nemenyi) ---
    n = len(modelos_ordenados)
    tamanho = max(12, n * 0.45)
    fig, ax = plt.subplots(figsize=(tamanho + 3, tamanho))

    cmap = ListedColormap(CORES_FAIXAS)
    norm = BoundaryNorm(FAIXAS, cmap.N)
    mask = np.triu(np.ones_like(p_valores_nemenyi, dtype=bool))

    sns.heatmap(
        p_valores_nemenyi.astype(float),
        mask=mask, cmap=cmap, norm=norm,
        square=True, linewidths=0.4, linecolor="white",
        annot=(n <= 25), fmt=".2f", annot_kws={"fontsize": 7},
        cbar=False, ax=ax,
    )

    for lab in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        g = grupo_metodo(lab.get_text())
        lab.set_color(COR_GRUPO[g])
        if g != "Literatura":
            lab.set_fontweight("bold")

    grupos_seq = [grupo_metodo(m) for m in modelos_ordenados]
    for i in range(1, n):
        if grupos_seq[i] != grupos_seq[i - 1]:
            ax.axhline(i, color="white", linewidth=4)
            ax.axvline(i, color="white", linewidth=4)

    patches = [Patch(facecolor=c, edgecolor="gray", label=r) for c, r in zip(CORES_FAIXAS, ROTULOS_FAIXAS)]
    ax.legend(handles=patches, title="Equivalência entre rankings (p-valor do Nemenyi)",
              loc="upper right", bbox_to_anchor=(1.0, 1.0), fontsize=10, title_fontsize=11, frameon=True)

    titulo_mapa = "Mapa de calor de equivalência estatística entre rankings (Nemenyi)"
    if classe_rotulo:
        titulo_mapa = f"{rotulo_exibicao} -- {titulo_mapa}"
    ax.set_title(f"{titulo_mapa}\nAzul = CEQ  |  Verde = Descritivas  |  Cinza = Literatura", fontsize=15, pad=20)
    plt.xticks(rotation=90)
    plt.tight_layout()

    file_out_plot = os.path.join(OUT_DIR, "3_Heatmap_Equivalencia_Rankings.png")
    plt.savefig(file_out_plot, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Mapa de calor salvo em: {file_out_plot}")
    print(f"Processo concluído para {rotulo_exibicao}! Tudo gerado em: {OUT_DIR}")


### 3. Execução — uma vez por subpasta de classe (ou pasta única)

In [ ]:
for pasta_escopo, classe_rotulo in ESCOPOS:
    processar_escopo(pasta_escopo, classe_rotulo)

print("\nAnálise de equivalência concluída para todos os escopos.")
